In [42]:
import numpy as np
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

In [43]:
import torchvision.datasets as datasets
mnist_trainset = datasets.MNIST(root='./data', train=True, download=False, transform=None)
mnist_testset = datasets.MNIST(root='./data', train=False, download=False, transform=None)

In [44]:
(xTrain, yTrain) = mnist_trainset.data.to(torch.float32), mnist_trainset.targets.to(torch.float32)
(xTest, yTest) = mnist_testset.data.to(torch.float32), mnist_testset.targets.to(torch.float32)
xTrain = xTrain.view(-1,28*28) / 255.0
xTest = xTest.view(-1,28*28) / 255.0

In [45]:
class SoftmaxRegressor(nn.Module):
    def __init__(self, in_features: int, n_classes: int): 
        self.in_features = in_features
        self.n_classes = n_classes
        super().__init__() # Calls init method of nn.Module for all instances of this class
        self.linear = torch.nn.Linear(in_features,n_classes)

    def forward(self,X):
        return self.linear(X)
    
    def predict_proba(self,X):
        logits = self.forward(X)
        probas = F.softmax(logits,1)
        return probas

In [46]:
model = SoftmaxRegressor(28*28, 10)
print(model.parameters)
optim = torch.optim.SGD(model.parameters(), lr=0.2)
loss_fn = nn.CrossEntropyLoss()
n_epoch = 100

<bound method Module.parameters of SoftmaxRegressor(
  (linear): Linear(in_features=784, out_features=10, bias=True)
)>


In [47]:
model.train()
for epoch in range(n_epoch):
    logits = model(xTrain)
    optim.zero_grad() # Sets the gradients to 0 for the fresh epoch
    # Here. Loss is given logits and labels due to definition of crossentropyloss.
    loss = loss_fn(logits,yTrain.long())
    loss.backward() # Computes all gradients necessary for training.
    optim.step() # optim.step() causes the weights to change according to derivatives computed above.
    print(f"Loss at step {epoch} is: {loss}")
model.eval()

Loss at step 0 is: 2.3165132999420166
Loss at step 1 is: 2.100050210952759
Loss at step 2 is: 1.92544686794281
Loss at step 3 is: 1.7758607864379883
Loss at step 4 is: 1.6476891040802002
Loss at step 5 is: 1.5379761457443237
Loss at step 6 is: 1.4438774585723877
Loss at step 7 is: 1.362838625907898
Loss at step 8 is: 1.2926703691482544
Loss at step 9 is: 1.2315442562103271
Loss at step 10 is: 1.177956461906433
Loss at step 11 is: 1.130678653717041
Loss at step 12 is: 1.0887097120285034
Loss at step 13 is: 1.0512336492538452
Loss at step 14 is: 1.0175827741622925
Loss at step 15 is: 0.9872088432312012
Loss at step 16 is: 0.9596586227416992
Loss at step 17 is: 0.934556245803833
Loss at step 18 is: 0.9115877747535706
Loss at step 19 is: 0.8904892206192017
Loss at step 20 is: 0.8710379600524902
Loss at step 21 is: 0.853044331073761
Loss at step 22 is: 0.8363467454910278
Loss at step 23 is: 0.8208065629005432
Loss at step 24 is: 0.8063036203384399
Loss at step 25 is: 0.7927344441413879
Loss

SoftmaxRegressor(
  (linear): Linear(in_features=784, out_features=10, bias=True)
)

In [48]:
xTest[0].size()

torch.Size([784])

In [49]:
probas = model.predict_proba(xTest)
pred = probas[3].detach().numpy()
print(torch.argmax(probas[3]), yTrain[3])

tensor(0) tensor(1.)
